# Material Candidate Explorer — True Closed-Loop Colab

[![Open in Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jujumelona/material-candidate-explorer/blob/main/MATERIAL_CANDIDATE_EXPLORER_V11_PROMPT_RAG_REAL_GENERATION.ipynb)

## Goal

This material-only notebook runs the repository's real adaptive search engine:

1. retrieve one source-grounded literature evidence bundle;
2. evaluate the parent crystal;
3. generate a MatterGen candidate batch;
4. re-evaluate every child with MatterSim and CHGNet;
5. update target-property, novelty, disagreement, Pareto, and stability branch pools;
6. adapt generation controls and feed selected children into the next round.

Optional RAG, scholarly API, and MCP fields may be left blank. Secrets are requested with hidden input and are never written into this notebook or the result ZIP.

Scientific boundary: ranked outputs are computational leads. They are not proof of thermodynamic stability, novelty, synthesizability, superconductivity, safety, or any medical property.

In [ ]:
# @title 0. Material search settings
DISCOVERY_PROMPT = "Find low-energy Li-O crystal candidates while preserving structural diversity and literature-linked failure modes." # @param {type:"string"}
CHEMICAL_SYSTEM = "Li-O" # @param {type:"string"}
RUN_LABEL = "li-o-material-search" # @param {type:"string"}

SEARCH_ROUNDS = 3 # @param {type:"integer"}
CANDIDATE_COUNT = 2 # @param {type:"integer"}
FRONTIER_WIDTH = 1 # @param {type:"integer"}
RANKING_LIMIT = 30 # @param {type:"integer"}
BASE_SEED = 20260716 # @param {type:"integer"}
MAX_SEARCH_HOURS = 6.0 # @param {type:"number"}
ENABLE_CONTROL_SWEEP = False # @param {type:"boolean"}

RUN_LIVE_RAG = True # @param {type:"boolean"}
RAG_FROM_DATE = "2024-01-01" # @param {type:"date"}
RAG_MAX_RESULTS = 15 # @param {type:"integer"}
RAG_MAX_BRANCHES = 20 # @param {type:"integer"}
CONTACT_EMAIL = "" # @param {type:"string"}

RAG_MODEL_API_URL = "" # @param {type:"string"}
RAG_MODEL_NAME = "" # @param {type:"string"}
MATERIAL_RAG_MCP_URL = "" # @param {type:"string"}
MATERIAL_RAG_MCP_TOOL = "" # @param {type:"string"}
MATERIAL_RAG_MCP_ALLOW_LOOPBACK_HTTP = False # @param {type:"boolean"}

INSTALL_OR_REUSE_MODELS = True # @param {type:"boolean"}
DOWNLOAD_RESULTS_ZIP = True # @param {type:"boolean"}

import re

if not DISCOVERY_PROMPT.strip():
    raise ValueError("DISCOVERY_PROMPT is required.")
if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9._-]{0,62}", RUN_LABEL.strip()):
    raise ValueError("RUN_LABEL must use letters, digits, dots, underscores, or hyphens.")
if not 1 <= SEARCH_ROUNDS <= 100:
    raise ValueError("SEARCH_ROUNDS must be between 1 and 100.")
if not 1 <= CANDIDATE_COUNT <= 1024:
    raise ValueError("CANDIDATE_COUNT must be between 1 and 1024.")
if not 1 <= FRONTIER_WIDTH <= 64:
    raise ValueError("FRONTIER_WIDTH must be between 1 and 64.")
if not 1 <= RANKING_LIMIT <= 1024:
    raise ValueError("RANKING_LIMIT must be between 1 and 1024.")
if not 0.1 <= MAX_SEARCH_HOURS <= 48:
    raise ValueError("MAX_SEARCH_HOURS must be between 0.1 and 48.")
if not 1 <= RAG_MAX_RESULTS <= 200:
    raise ValueError("RAG_MAX_RESULTS must be between 1 and 200.")
if not 1 <= RAG_MAX_BRANCHES <= 200:
    raise ValueError("RAG_MAX_BRANCHES must be between 1 and 200.")
if bool(RAG_MODEL_API_URL.strip()) != bool(RAG_MODEL_NAME.strip()):
    raise ValueError("RAG_MODEL_API_URL and RAG_MODEL_NAME must be filled together.")
if bool(MATERIAL_RAG_MCP_URL.strip()) != bool(MATERIAL_RAG_MCP_TOOL.strip()):
    raise ValueError("MATERIAL_RAG_MCP_URL and MATERIAL_RAG_MCP_TOOL must be filled together.")

print("Settings accepted. Candidate batches and adaptive rounds are enabled.")

## Setup

Run the next two cells once. The first keeps credentials out of notebook source. The second installs the public repository, downloads pinned MatterGen weights, starts isolated sidecars, and imports the generated endpoint/revision environment.

In [ ]:
# @title 1. Enter optional secrets with hidden input
import getpass
import os

def optional_secret(environment_name: str, label: str) -> None:
    if os.environ.get(environment_name, "").strip():
        print(f"{label}: using an existing environment secret.")
        return
    value = getpass.getpass(f"{label} (optional; press Enter to skip): ").strip()
    if value:
        os.environ[environment_name] = value

if CONTACT_EMAIL.strip():
    os.environ["LITERATURE_CONTACT_EMAIL"] = CONTACT_EMAIL.strip()
    os.environ["LITERATURE_USER_AGENT"] = (
        f"DiscoveryOS/0.4 {CONTACT_EMAIL.strip()}"
    )

if RUN_LIVE_RAG:
    optional_secret("NCBI_API_KEY", "NCBI API key")
    optional_secret("OPENALEX_API_KEY", "OpenAlex API key")

    if RAG_MODEL_API_URL.strip():
        os.environ["RAG_MODEL_API_URL"] = RAG_MODEL_API_URL.strip()
        os.environ["RAG_MODEL_NAME"] = RAG_MODEL_NAME.strip()
        optional_secret("RAG_MODEL_API_KEY", "RAG model API key")

    if MATERIAL_RAG_MCP_URL.strip():
        os.environ["MATERIAL_RAG_MCP_URL"] = MATERIAL_RAG_MCP_URL.strip()
        os.environ["MATERIAL_RAG_MCP_TOOL"] = MATERIAL_RAG_MCP_TOOL.strip()
        os.environ["MATERIAL_RAG_MCP_ALLOW_LOOPBACK_HTTP"] = (
            "1" if MATERIAL_RAG_MCP_ALLOW_LOOPBACK_HTTP else "0"
        )
        optional_secret("MATERIAL_RAG_MCP_TOKEN", "MCP bearer token")

print("Secret configuration complete. Values were not printed.")

In [ ]:
# @title 2. Install the public project and start material sidecars
from __future__ import annotations

import json
import os
import shlex
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

REPOSITORY_URL = "https://github.com/jujumelona/material-candidate-explorer.git"
REPOSITORY_DIR = Path("/content/material-candidate-explorer")
RUN_ID = f"{RUN_LABEL.strip()}-{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"
RUN_ROOT = Path("/content") / RUN_ID
RUN_ROOT.mkdir(parents=True, exist_ok=False)

def run_checked(command: list[str], *, cwd: Path | None = None) -> subprocess.CompletedProcess:
    print("$", " ".join(shlex.quote(item) for item in command))
    return subprocess.run(command, cwd=cwd, check=True, text=True)

if not (3, 10) <= sys.version_info[:2] < (3, 13):
    raise RuntimeError(
        f"The public package supports Python 3.10-3.12; this runtime is {sys.version.split()[0]}."
    )
if shutil.which("nvidia-smi") is None:
    raise RuntimeError(
        "A CUDA Colab runtime is required. Select Runtime > Change runtime type > T4 GPU."
    )

if not (REPOSITORY_DIR / ".git").is_dir():
    run_checked(["git", "clone", "--depth", "1", REPOSITORY_URL, str(REPOSITORY_DIR)])
else:
    dirty = subprocess.run(
        ["git", "-C", str(REPOSITORY_DIR), "status", "--porcelain", "--untracked-files=no"],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
    if dirty:
        raise RuntimeError(
            "The dedicated public clone has local tracked edits. Restart the runtime or preserve them before updating."
        )
    run_checked(["git", "-C", str(REPOSITORY_DIR), "fetch", "--depth", "1", "origin", "main"])
    run_checked(["git", "-C", str(REPOSITORY_DIR), "checkout", "main"])
    run_checked(["git", "-C", str(REPOSITORY_DIR), "merge", "--ff-only", "origin/main"])

run_checked([sys.executable, "-m", "pip", "install", "-e", "."], cwd=REPOSITORY_DIR)

os.environ.update(
    {
        "MATTERGEN_PRETRAINED_NAME": "chemical_system_energy_above_hull",
        "MATTERGEN_DEVICE": "cuda",
        "MATTERSIM_DEVICE": "cpu",
        "CHGNET_DEVICE": "cpu",
        "MATTERSIM_WEIGHT_REVISION": (
            "managed-unattested:mattersim-upstream-managed"
        ),
        "CHGNET_WEIGHT_REVISION": (
            "managed-unattested:chgnet-0.3.0-builtin"
        ),
    }
)

if INSTALL_OR_REUSE_MODELS:
    run_checked(
        [
            "bash",
            "bootstrap.sh",
            "--profile",
            "materials-open",
            "--accelerator",
            "cuda",
            "--include-weights",
        ],
        cwd=REPOSITORY_DIR,
    )
    run_checked(
        [
            "bash",
            "start-sidecars.sh",
            "--profile",
            "materials-open",
            "--component",
            "mattergen",
            "--component",
            "mattersim",
            "--component",
            "chgnet",
            "--ready-timeout-seconds",
            "900",
        ],
        cwd=REPOSITORY_DIR,
    )

environment_file = REPOSITORY_DIR / ".discovery" / "sidecars.env.sh"
if not environment_file.is_file():
    raise RuntimeError(
        f"{environment_file} was not created. Run the model installation/start step."
    )

loaded_environment = subprocess.run(
    [
        "bash",
        "-lc",
        f"set -a; . {shlex.quote(str(environment_file))}; env -0",
    ],
    check=True,
    capture_output=True,
).stdout
for entry in loaded_environment.split(b"\0"):
    if b"=" not in entry:
        continue
    key, value = entry.split(b"=", 1)
    os.environ[key.decode()] = value.decode()

import requests

health_rows = []
for component, variable in (
    ("MatterGen", "MATTERGEN_API_URL"),
    ("MatterSim", "MATTERSIM_API_URL"),
    ("CHGNet", "CHGNET_API_URL"),
):
    endpoint = os.environ.get(variable, "").rstrip("/")
    if not endpoint:
        raise RuntimeError(f"{variable} is missing from the generated sidecar environment.")
    response = requests.get(f"{endpoint}/health", timeout=30)
    response.raise_for_status()
    health_rows.append(
        {
            "component": component,
            "endpoint": endpoint,
            "status": response.json().get("status", "ready"),
        }
    )

print(json.dumps(health_rows, indent=2))
print("Run artifacts:", RUN_ROOT)

## Step 1 — Build strict material-search inputs

The two required selection objectives are properties actually returned by both material experts: energy per atom and maximum force. Energy above hull remains an optional MatterGen generation condition. Marking it as a required expert result would exclude every candidate and can make a search stop after one round.

In [ ]:
# @title 3. Create goal.json, parent.json, and run-config.json
import math
import re

from discovery_os.fusion_schemas import (
    GenerationControls,
    WorkspaceMode,
    WorkspaceRunConfig,
)
from discovery_os.hashing import candidate_content_hash, stable_hash
from discovery_os.integration_manifest import load_integration_manifest
from discovery_os.schemas import (
    Candidate,
    CandidateRef,
    CandidateRepresentation,
    CandidateType,
    DiscoveryDomain,
    DiscoveryGoal,
    ObjectiveDirection,
    PropertyObjective,
    RepresentationKind,
)

PERIODIC_TABLE = set(
    "H He Li Be B C N O F Ne Na Mg Al Si P S Cl Ar K Ca Sc Ti V Cr Mn Fe Co Ni "
    "Cu Zn Ga Ge As Se Br Kr Rb Sr Y Zr Nb Mo Tc Ru Rh Pd Ag Cd In Sn Sb Te I "
    "Xe Cs Ba La Ce Pr Nd Pm Sm Eu Gd Tb Dy Ho Er Tm Yb Lu Hf Ta W Re Os Ir Pt "
    "Au Hg Tl Pb Bi Po At Rn Fr Ra Ac Th Pa U Np Pu Am Cm Bk Cf Es Fm Md No Lr "
    "Rf Db Sg Bh Hs Mt Ds Rg Cn Nh Fl Mc Lv Ts Og".split()
)

symbols = [
    item
    for item in re.split(r"[-,\s]+", CHEMICAL_SYSTEM.strip())
    if item
]
if not 1 <= len(symbols) <= 16 or any(item not in PERIODIC_TABLE for item in symbols):
    raise ValueError(
        "CHEMICAL_SYSTEM must contain 1-16 element symbols separated by hyphens, for example Li-O."
    )
chemical_system = "-".join(dict.fromkeys(symbols))

def make_seed_cif(elements: list[str]) -> str:
    grid = (0.0, 0.25, 0.5, 0.75)
    atom_rows = []
    for index, element in enumerate(elements):
        x = grid[index % 4]
        y = grid[(index // 4) % 4]
        z = grid[(index // 16) % 4]
        atom_rows.append(
            f"{element}{index + 1} {element} {x:.4f} {y:.4f} {z:.4f} 1.0"
        )
    return "\n".join(
        [
            "data_material_search_seed",
            "_symmetry_space_group_name_H-M 'P 1'",
            "_symmetry_Int_Tables_number 1",
            "_cell_length_a 8.000",
            "_cell_length_b 8.000",
            "_cell_length_c 8.000",
            "_cell_angle_alpha 90.000",
            "_cell_angle_beta 90.000",
            "_cell_angle_gamma 90.000",
            "loop_",
            "_atom_site_label",
            "_atom_site_type_symbol",
            "_atom_site_fract_x",
            "_atom_site_fract_y",
            "_atom_site_fract_z",
            "_atom_site_occupancy",
            *atom_rows,
            "",
        ]
    )

goal = DiscoveryGoal(
    goal_id=f"{RUN_LABEL.strip()}-goal",
    domain=DiscoveryDomain.GENERAL_MATERIALS,
    title=f"{chemical_system} adaptive crystal candidate search",
    scientific_question=DISCOVERY_PROMPT.strip(),
    objectives=[
        PropertyObjective(
            property_name="energy_per_atom",
            direction=ObjectiveDirection.MINIMIZE,
            unit="eV/atom",
            weight=1.0,
            required=True,
            rationale="Comparable energetic diagnostic returned by MatterSim and CHGNet.",
        ),
        PropertyObjective(
            property_name="max_force",
            direction=ObjectiveDirection.MINIMIZE,
            unit="eV/angstrom",
            weight=0.5,
            required=True,
            rationale="Reject highly unrelaxed or collapsed structures.",
        ),
        PropertyObjective(
            property_name="energy_above_hull",
            direction=ObjectiveDirection.TARGET,
            unit="eV/atom",
            target_value=0.0,
            weight=0.0,
            required=False,
            rationale="MatterGen condition only; it is not returned by the primary expert panel.",
        ),
    ],
    validation_profile_id="general-materials-screening-v1",
    candidate_types=[CandidateType.CRYSTAL],
    assumptions=[
        "The initial CIF is a lineage seed, not a claimed stable phase.",
        "Search ranks are diagnostic and require higher-fidelity validation.",
    ],
    max_cycles=SEARCH_ROUNDS,
)

draft_parent = Candidate(
    candidate_id=f"{RUN_LABEL.strip()}-seed",
    candidate_type=CandidateType.CRYSTAL,
    domain=DiscoveryDomain.GENERAL_MATERIALS,
    name=f"{chemical_system} lineage seed",
    representations=[
        CandidateRepresentation(
            kind=RepresentationKind.CIF,
            value=make_seed_cif(symbols),
            media_type="chemical/x-cif",
            format_version="CIF 1.1",
            canonical=False,
        )
    ],
    attributes={
        "chemical_system": chemical_system,
        "seed_only": True,
    },
    provenance={"role": "lineage_seed"},
)
parent_payload = draft_parent.model_dump(mode="python")
parent_payload["candidate_ref"] = CandidateRef(
    candidate_id=draft_parent.candidate_id,
    version=1,
    content_hash=candidate_content_hash(draft_parent),
)
parent = Candidate.model_validate(parent_payload)

controls = GenerationControls(
    alpha=0.5,
    temperature=1.0,
    mutation_strength=0.2,
    diversity_strength=0.3,
    schedule_step=0,
    decision_reason="initial Colab material-search controls",
)

manifest = load_integration_manifest()
mattergen = next(
    component
    for component in manifest.components
    if component.component_id == "mattergen"
)
mattergen_weight_revision = os.environ.get(
    "MATTERGEN_WEIGHT_REVISION",
    mattergen.weights[0].revision,
).strip()

run_config = WorkspaceRunConfig(
    workspace_mode=WorkspaceMode.ON,
    seed=BASE_SEED,
    generator_seed=BASE_SEED,
    goal_hash=stable_hash(goal),
    parent_candidate_ref=parent.candidate_ref,
    pair_key=f"{RUN_LABEL.strip()}-closed-loop",
    cohort_index=0,
    generator_id="mattergen",
    generator_version=mattergen.install.version,
    generator_code_revision=mattergen.source.revision,
    generator_weight_revision=mattergen_weight_revision,
    generator_parameters_hash=stable_hash(
        {
            "pretrained_name": os.environ["MATTERGEN_PRETRAINED_NAME"],
            "chemical_system": chemical_system,
            "controls": controls,
        }
    ),
    decoder_config_hash=stable_hash({"decoder": "mattergen-sidecar-v1"}),
    postprocessing_hash=stable_hash({"deduplication": "exact-content-v1"}),
    resource_budget_hash=stable_hash(
        {
            "runtime": "colab-cuda",
            "max_search_hours": MAX_SEARCH_HOURS,
        }
    ),
    evaluator_panel_hash=stable_hash(
        {"experts": ["mattersim", "chgnet"]}
    ),
    candidate_count=CANDIDATE_COUNT,
    generation_controls=controls,
)

GOAL_PATH = RUN_ROOT / "goal.json"
PARENT_PATH = RUN_ROOT / "parent.json"
RUN_CONFIG_PATH = RUN_ROOT / "run-config.json"
GOAL_PATH.write_text(goal.model_dump_json(indent=2), encoding="utf-8")
PARENT_PATH.write_text(parent.model_dump_json(indent=2), encoding="utf-8")
RUN_CONFIG_PATH.write_text(run_config.model_dump_json(indent=2), encoding="utf-8")

print("Chemical system:", chemical_system)
print("Goal hash:", run_config.goal_hash)
print("Candidate batch size:", run_config.candidate_count)
print("Input directory:", RUN_ROOT)

## Step 2 — Retrieve one evidence bundle

RAG queries configured scholarly sources and the optional MCP tool once. The resulting immutable bundle is passed into the search. During every round, the evidence policy assigns branches and updates their priorities from measured improvement, structural collapse, and failures; it does not repeatedly query the network and discard prior evidence.

In [ ]:
# @title 4. Run source-grounded literature RAG
from datetime import date

import pandas as pd
from IPython.display import display

from discovery_os.literature_rag import (
    JsonEvidenceIndex,
    LiteratureSource,
    build_literature_rag_from_environment,
    save_evidence_bundle,
)

RAG_BUNDLE_PATH = None
if RUN_LIVE_RAG:
    from_date = date.fromisoformat(RAG_FROM_DATE) if RAG_FROM_DATE else None
    rag_pipeline = build_literature_rag_from_environment(
        require_model=bool(RAG_MODEL_API_URL.strip())
    )
    rag_bundle = rag_pipeline.run(
        DISCOVERY_PROMPT,
        goal=goal,
        sources=list(LiteratureSource),
        from_date=from_date,
        max_results_per_query=RAG_MAX_RESULTS,
        max_branches=RAG_MAX_BRANCHES,
        index=JsonEvidenceIndex(RUN_ROOT / "evidence-index"),
    )
    RAG_BUNDLE_PATH = RUN_ROOT / "literature-evidence.json"
    save_evidence_bundle(rag_bundle, RAG_BUNDLE_PATH)

    source_rows = [
        {
            "source": str(item.source),
            "status": str(item.status),
            "results": item.result_count,
            "queries": len(item.query_ids),
            "seconds": round(item.elapsed_seconds, 2),
            "error": item.error,
        }
        for item in rag_bundle.source_statuses
    ]
    display(pd.DataFrame(source_rows))
    print(
        f"RAG kept {len(rag_bundle.records)} records, "
        f"{len(rag_bundle.claims)} source-grounded claims, and "
        f"{len(rag_bundle.branches)} search-prior branches."
    )
    for warning in rag_bundle.warnings[:10]:
        print("RAG warning:", warning)
else:
    print("Live RAG is off. The material search will run without a literature policy.")

## Step 3 — Run the real adaptive loop

For a Colab T4, the default uses one frontier candidate per branch and disables the extra alpha/temperature grid. The adaptive scheduler still changes controls after each completed round. Enabling the control sweep multiplies generator calls and GPU time.

In [ ]:
# @title 5. Generate, re-evaluate, select, and repeat
import subprocess
import time

ARTIFACT_ROOT = RUN_ROOT / "fusion-search"
SEARCH_REPORT_PATH = RUN_ROOT / "fusion-search-cli-report.json"

command = [
    sys.executable,
    "-m",
    "discovery_os.cli",
    "fusion-search",
    "--search-id",
    RUN_ID,
    "--goal",
    str(GOAL_PATH),
    "--parent",
    str(PARENT_PATH),
    "--run-config",
    str(RUN_CONFIG_PATH),
    "--generator",
    "mattergen",
    "--rounds",
    str(SEARCH_ROUNDS),
    "--frontier-width",
    str(FRONTIER_WIDTH),
    "--ranking-limit",
    str(RANKING_LIMIT),
    "--modality",
    "crystal_material",
    "--expert",
    "mattersim",
    "--expert",
    "chgnet",
    "--required-evaluator",
    "mattersim",
    "--required-evaluator",
    "chgnet",
    "--artifacts",
    str(ARTIFACT_ROOT),
]
if not ENABLE_CONTROL_SWEEP:
    command.append("--no-control-sweep")
if RAG_BUNDLE_PATH is not None:
    command.extend(["--rag-bundle", str(RAG_BUNDLE_PATH)])

print("Starting the closed loop. This can take hours for several rounds.")
started = time.monotonic()
try:
    completed = subprocess.run(
        command,
        cwd=REPOSITORY_DIR,
        capture_output=True,
        text=True,
        timeout=MAX_SEARCH_HOURS * 3600,
    )
except subprocess.TimeoutExpired as exc:
    raise RuntimeError(
        f"Search exceeded {MAX_SEARCH_HOURS} hours. Sidecar logs are under "
        f"{REPOSITORY_DIR / '.discovery' / 'logs' / 'sidecars'}."
    ) from exc

if completed.stderr.strip():
    (RUN_ROOT / "fusion-search-stderr.log").write_text(
        completed.stderr,
        encoding="utf-8",
    )
if completed.returncode != 0:
    print(completed.stderr[-6000:])
    raise RuntimeError(
        f"fusion-search failed with exit code {completed.returncode}. "
        f"See {RUN_ROOT / 'fusion-search-stderr.log'}."
    )

SEARCH_PAYLOAD = json.loads(completed.stdout)
SEARCH_REPORT_PATH.write_text(
    json.dumps(SEARCH_PAYLOAD, indent=2),
    encoding="utf-8",
)
report = SEARCH_PAYLOAD["report"]
print(
    f"Search status={report['status']}; "
    f"rounds={report['rounds_completed']}/{report['rounds_requested']}; "
    f"cycles={len(report['cycle_records'])}; "
    f"ranked_candidates={len(report['ranked_candidates'])}; "
    f"elapsed_minutes={(time.monotonic() - started) / 60:.1f}"
)

## Checks and results

The tables below expose the actual round history, five frontiers, scheduler decisions, expert property vectors, and failures. If a search stops early, the notebook reports that state instead of pretending that all requested rounds completed.

In [ ]:
# @title 6. Inspect rounds, controls, rankings, and export CIF files
import re
import shutil

expected_branches = {
    "stability",
    "target_property",
    "novelty",
    "expert_disagreement",
    "pareto",
}
reported_branches = {str(item["branch"]) for item in report["branches"]}
if reported_branches != expected_branches:
    raise RuntimeError(
        f"Search report did not preserve all branch pools: {sorted(reported_branches)}"
    )

round_rows = []
for item in report["round_history"]:
    row = {
        "round": item["round_index"],
        "cycles": len(item["cycle_record_ids"]),
        "generated_records": len(item["candidate_record_ids"]),
        "failures": len(item["failure_record_ids"]),
    }
    for branch_name, record_ids in item["branch_frontiers"].items():
        row[f"frontier.{branch_name}"] = len(record_ids)
    round_rows.append(row)

print("Actual round history")
display(pd.DataFrame(round_rows))

scheduler_rows = []
for branch in report["branches"]:
    for step, decision in enumerate(branch["scheduler_history"], start=1):
        scheduler_rows.append(
            {
                "branch": str(branch["branch"]),
                "step": step,
                "improvement": decision["observation"]["objective_improvement"],
                "collapse_rate": decision["observation"]["structural_collapse_rate"],
                "alpha": decision["controls"]["alpha"],
                "temperature": decision["controls"]["temperature"],
                "mutation": decision["controls"]["mutation_strength"],
                "diversity": decision["controls"]["diversity_strength"],
                "reason": "; ".join(decision["reasons"]),
            }
        )

print("Adaptive scheduler history")
if scheduler_rows:
    display(pd.DataFrame(scheduler_rows))
else:
    print("No scheduler decision was recorded. Inspect failures and round history.")

ranking_rows = []
CIF_DIRECTORY = RUN_ROOT / "ranked-cifs"
CIF_DIRECTORY.mkdir(parents=True, exist_ok=True)

for ranked in report["ranked_candidates"]:
    row = {
        "rank": ranked["rank"],
        "candidate_id": ranked["candidate_ref"]["candidate_id"],
        "source_round": ranked["source_round"],
        "source_branch": str(ranked["source_branch"]),
        "priority_score": ranked["priority_score"],
        "pareto_member": ranked["pareto_member"],
        "expert_disagreement": ranked["expert_disagreement_score"],
        "mean_uncertainty": ranked["mean_reported_uncertainty"],
        "alpha": ranked["generation_controls"]["alpha"],
        "temperature": ranked["generation_controls"]["temperature"],
        "branch_ranks": json.dumps(ranked["branch_ranks"], sort_keys=True),
    }
    for expert_id, properties in ranked["expert_property_vectors"].items():
        for property_row in properties:
            unit = property_row.get("unit") or "unitless"
            column = f"{expert_id}.{property_row['property_name']} [{unit}]"
            row[column] = property_row["value"]

    cif_value = next(
        (
            representation["value"]
            for representation in ranked["candidate"]["representations"]
            if str(representation["kind"]) == "cif"
        ),
        None,
    )
    if cif_value:
        safe_candidate_id = re.sub(
            r"[^A-Za-z0-9._-]+",
            "_",
            ranked["candidate_ref"]["candidate_id"],
        )[:120]
        cif_path = CIF_DIRECTORY / (
            f"{ranked['rank']:03d}-{safe_candidate_id}.cif"
        )
        cif_path.write_text(cif_value, encoding="utf-8")
        row["cif_file"] = str(cif_path.relative_to(RUN_ROOT))
    ranking_rows.append(row)

ranking_table = pd.DataFrame(ranking_rows)
print("Unified evidence-preserving candidate ranking")
display(ranking_table.head(RANKING_LIMIT))
ranking_table.to_csv(RUN_ROOT / "ranked-candidates.csv", index=False)

if report["rounds_completed"] < report["rounds_requested"]:
    print(
        "Search stopped before all requested rounds. "
        "This is an explicit exhausted, partial, or failed state."
    )
    failure_rows = [
        {
            "round": item["round_index"],
            "branch": str(item["branch"]),
            "cause_type": item["cause_type"],
            "cause": item["cause"],
        }
        for item in report["failure_records"]
    ]
    if failure_rows:
        display(pd.DataFrame(failure_rows).head(100))

archive_path = Path(
    shutil.make_archive(
        str(RUN_ROOT),
        "zip",
        root_dir=RUN_ROOT,
    )
)
print("Result ZIP:", archive_path)
print("Validation handoff candidates:", len(report["validation_handoff_candidate_refs"]))

if DOWNLOAD_RESULTS_ZIP:
    try:
        from google.colab import files
    except ImportError:
        print("Automatic download is available in Google Colab only.")
    else:
        files.download(str(archive_path))

## Next steps

Use "validation_handoff_candidate_refs" only as a bounded shortlist. Before any scientific claim, add structure standardization, relaxation, reference phases and a validated convex-hull calculation, phonons or other task-specific physics, synthesis planning, and independent experimental validation.

For larger GPUs, increase frontier width or enable the control sweep gradually. Increasing both frontier width and candidate count multiplies MatterGen and expert calls across all five branches.